In [ ]:
import numpy as np

# Define a small 5x5 land cover map for simulation
lc_array = np.array([
    [1, 0, 0, 0, 2],
    [3, 4, 4, 4, 5],
    [6, 4, 4, 4, 8],
    [7, 4, 4, 4, 1],
    [2, 3, 6, 7, 1]
])

# Define forest value
forest_val = 4

# Initialize edge encoding array
edge_encoding = np.zeros_like(lc_array, dtype=np.uint8)

# Determine edge encoding based on forest value
rows, cols = lc_array.shape
for i in range(1, rows - 1):
    for j in range(1, cols - 1):
        if lc_array[i, j] == forest_val:
            code = 0b0000
            if lc_array[i - 1, j] != forest_val:
                code |= 0b1000
            if lc_array[i + 1, j] != forest_val:
                code |= 0b0100
            if lc_array[i, j - 1] != forest_val:
                code |= 0b0010
            if lc_array[i, j + 1] != forest_val:
                code |= 0b0001
            edge_encoding[i, j] = code

# Generate final encoded map that includes landcover
encoded_final_map = np.zeros_like(lc_array, dtype=np.uint16)
for i in range(1, rows - 1):
    for j in range(1, cols - 1):
        code = edge_encoding[i, j]
        if code > 0:  # It's an edge pixel
            top = lc_array[i - 1, j] if code & 0b1000 else 0
            bottom = lc_array[i + 1, j] if code & 0b0100 else 0
            left = lc_array[i, j - 1] if code & 0b0010 else 0
            right = lc_array[i, j + 1] if code & 0b0001 else 0
            encoded_final_map[i, j] = 1*10000 + top*1000 + bottom*100 + left*10 + right

# Display the results
edge_encoding, encoded_final_map


In [ ]:
import numpy as np
import rasterio
from tqdm import tqdm
import os
import gc

def process_year(year):
    edge_file = f"G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\LCMAP_edges\\LCMAP_{year}_edges.tif"
    landcover_file = f"G:\\Public\\LCMAP\\LCMAP_CU_{year}_V13_LCPRI.tif"
    output_file = f"G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Edge_adjunct_LC\\Adjunct_LC_{year}.tif"
    print(year)

    with rasterio.open(edge_file) as edge_src, rasterio.open(landcover_file) as lc_src:
        edge_array = edge_src.read(1)
        lc_array = lc_src.read(1)
        rows, cols = lc_array.shape
        encoded_final_map = np.zeros_like(lc_array, dtype=np.uint16)
        
        for i in tqdm(range(1, rows - 1)):
            for j in range(1, cols - 1):
                code = edge_array[i, j]
                if code > 0:  # It's an edge pixel
                    top = lc_array[i - 1, j] if code & 0b1000 else 0
                    bottom = lc_array[i + 1, j] if code & 0b0100 else 0
                    left = lc_array[i, j - 1] if code & 0b0010 else 0
                    right = lc_array[i, j + 1] if code & 0b0001 else 0
                    encoded_final_map[i, j] = 1*10000 + top*1000 + bottom*100 + left*10 + right
        del edge_array
        del lc_array
        gc.collect()
        # Save the final encoded map
        with rasterio.open(output_file, 'w', driver='GTiff', height=encoded_final_map.shape[0],
                           width=encoded_final_map.shape[1], count=1, dtype='uint16',
                           crs=lc_src.crs, transform=lc_src.transform) as dst:
            dst.write(encoded_final_map, 1)
        del encoded_final_map
        gc.collect()

# Process each year from 1985 to 2021
for year in (range(1995, 2001)):
    process_year(year)
